# Module 6: individual data cleaning
Student: chenwang266 | branch: chenwang.

Working question: how did Los Angeles neighborhood/room-type supply and advertised prices change between March and June? Team confirmation remains pending. Proposed rules are in the decision log. Restart Kernel and Run All.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
candidates=[Path.cwd(),Path.cwd()/'module6_submission',Path.cwd().parent]
PACKAGE=next(p.resolve() for p in candidates if (p/'code'/'clean_data.py').exists())
SOURCE_DIR=PACKAGE.parent # edit if original CSVs live elsewhere
OUTPUT_DIR=PACKAGE/'data_and_evidence'
sys.path.insert(0,str(PACKAGE/'code'))
from clean_data import run
from validate_data import validate

## Inspect original data and proposed rules
A row is one listing in one snapshot. IDs stay strings. Originals remain unchanged.

In [2]:
inspection=[]
for filename in ['listings 16 March, 2026.csv','listings 15 June, 2026.csv']:
    raw=pd.read_csv(SOURCE_DIR/filename,dtype='string',keep_default_na=False)
    inspection.append({'file':filename,'rows':len(raw),'columns':len(raw.columns),'exact_duplicates':int(raw.duplicated().sum()),'repeated_ids':int(raw.id.duplicated().sum()),'missing_price':int(raw.price.eq('').sum())})
display(pd.DataFrame(inspection))
print((PACKAGE/'report'/'cleaning_decision_log.md').read_text())

,file,rows,columns,exact_duplicates,repeated_ids,missing_price
0,"listings 16 March, 2026.csv",46088,90,0,0,8910
1,"listings 15 June, 2026.csv",43751,90,0,0,5565


# Proposed cleaning rules - written before running the pipeline

Author: chenwang266. Branch: chenwang. Date: October 5, 2026.

Working vision: a short-term rental marketplace analyst decides which Los Angeles neighborhoods warrant follow-up pricing or host-support research. Question: how do listing counts, advertised median prices, calendar availability, and review ratings differ between the March and June 2026 snapshots, by neighborhood and room type? This is an individual working vision, awaiting team agreement.

One row is one listing observed in one snapshot. The expected key is (snapshot_date, id); repeated IDs between snapshots are valid observations.

1. Preserve both source CSVs byte-for-byte; hash them in the audit. Load every original column as text first, preserving identifier precision and leading zeros. Use the filename date as snapshot_date while retaining actual last_scraped dates.
2. Select 28 original fields needed for this analysis (see KEEP in clean_data.py). Exclud

## Clean full data and independently validate
Missing values remain missing. Extreme positive prices are retained and flagged.

In [3]:
results=run(SOURCE_DIR,OUTPUT_DIR)
validation=validate(SOURCE_DIR,OUTPUT_DIR)

{
  "rows": 89839,
  "merge": {
    "march_rows": 46088,
    "june_rows": 43751,
    "outer_rows": 51737,
    "matched": 38102,
    "march_only": 7986,
    "june_only": 5649,
    "matched_with_both_prices": 31893,
    "paired_median_price_change": 0.09999999999999432
  },
  "record_checks": "5/5 passed"
}


{
  "status": "PASS",
  "records_checked": 89839,
  "checks": [
    "source/output row and ID reconciliation",
    "all exported prices equal independently parsed source prices",
    "numeric bounds",
    "outer-merge union and intersection",
    "group and price denominators",
    "sample size, columns, and key membership",
    "five record results"
  ]
}


## Evidence, denominators, and five record checks
The sample supports inspection only; these audits cover the full data.

In [4]:
display(pd.read_csv(OUTPUT_DIR/'column_audit.csv'))
display(pd.read_csv(OUTPUT_DIR/'neighbourhood_metrics.csv').head(10))
print(json.dumps(results['merge'],indent=2))
display(pd.DataFrame(json.loads((OUTPUT_DIR/'record_checks.json').read_text())))
display(pd.read_csv(OUTPUT_DIR/'sample_cleaned_50.csv',dtype={'id':'string'}).head(10))

,snapshot_date,column,input_type,output_type,missing_before,missing_after,conversion_failures,whitespace_changes
0,2026-03-16,id,string,string,0,0,0,0
1,2026-03-16,neighbourhood_cleansed,string,string,0,0,0,0
2,2026-03-16,neighbourhood_group_cleansed,string,string,0,0,0,0
3,2026-03-16,property_type,string,string,0,0,0,0
4,2026-03-16,room_type,string,string,0,0,0,0
5,2026-03-16,accommodates,string,Float64,0,0,0,0
6,2026-03-16,bathrooms,string,Float64,10288,10288,0,0
7,2026-03-16,bedrooms,string,Float64,3196,3196,0,0
8,2026-03-16,beds,string,Float64,10439,10439,0,0
9,2026-03-16,price,string,Float64,8910,8910,0,0


,snapshot_date,neighbourhood_cleansed,room_type,listings,price_n,median_price,mean_price,availability_n,mean_availability_365,rating_n,mean_rating
0,2026-03-16,Acton,Entire home/apt,17,14,285.500,450.637857,17,225.588235,13,4.920769
1,2026-03-16,Acton,Private room,3,3,210.000,186.333333,3,364.333333,2,4.985000
2,2026-03-16,Adams-Normandie,Entire home/apt,23,19,165.370,221.215789,23,229.260870,15,4.834667
3,2026-03-16,Adams-Normandie,Private room,66,55,53.880,58.940182,66,211.303030,32,4.701875
4,2026-03-16,Agoura Hills,Entire home/apt,42,39,368.730,430.253846,42,255.452381,31,4.821290
5,2026-03-16,Agoura Hills,Private room,12,10,101.665,113.174000,12,208.666667,11,4.829091
6,2026-03-16,Agua Dulce,Entire home/apt,18,16,240.500,528.725625,18,258.277778,14,4.973571
7,2026-03-16,Agua Dulce,Private room,8,8,103.500,120.250000,8,273.125000,6,4.878333
8,2026-03-16,Alhambra,Entire home/apt,502,451,247.000,283.929712,502,211.521912,452,4.771814
9,2026-03-16,Alhambra,Private room,235,186,88.000,114.259247,235,191.451064,190,4.719579


{
  "march_rows": 46088,
  "june_rows": 43751,
  "outer_rows": 51737,
  "matched": 38102,
  "march_only": 7986,
  "june_only": 5649,
  "matched_with_both_prices": 31893,
  "paired_median_price_change": 0.09999999999999432
}


,case,snapshot_date,id,source_row,field,original,expected,actual,reason,matches
0,missing price,2026-03-16,1452068118494812958,2,price,,None,None,retain missing; never invent a zero,True
1,currency price,2026-03-16,1452093968341982976,3,price,$315.00,315.0,315.0,remove dollar sign and thousands separator,True
2,missing rating,2026-03-16,1453392869063399506,8,review_scores_rating,,None,None,retain missing; absence of rating is not a zer...,True
3,boolean label,2026-03-16,1452149405821816967,4,host_is_superhost,f,False,False,f means False,True
4,high advertised price,2026-03-16,98284,167,price,"$2,081.00",2081.0,2081.0,retain extreme positive price and flag it for ...,True


,id,neighbourhood_cleansed,neighbourhood_group_cleansed,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price,...,invalid_reviews_per_month,flag_high_price,flag_long_minimum_stay,flag_night_order,flag_availability_order,flag_review_order,flag_review_after_scrape,flag_unknown_room_type,price_missing,eligible_price_analysis
0,1452068118494812958,Echo Park,City of Los Angeles,Entire rental unit,Entire home/apt,2.0,NaN,1.0,NaN,NaN,...,False,False,False,False,False,False,False,False,True,False
1,1452093968341982976,Santa Monica,Other Cities,Entire rental unit,Entire home/apt,2.0,1.0,0.0,1.0,315.00,...,False,False,False,False,False,False,False,False,False,True
2,1453392869063399506,Hancock Park,City of Los Angeles,Entire home,Entire home/apt,15.0,8.0,7.0,8.0,5061.00,...,False,True,False,False,False,False,False,False,False,True
3,1452149405821816967,Venice,City of Los Angeles,Entire rental unit,Entire home/apt,5.0,1.0,2.0,2.0,204.80,...,False,False,False,False,False,False,False,False,False,True
4,98284,Sherman Oaks,City of Los Angeles,Entire home,Entire home/apt,10.0,6.5,6.0,6.0,2081.00,...,False,True,False,False,False,False,False,False,False,True
5,1079335,West Hollywood,Other Cities,Entire rental unit,Entire home/apt,2.0,1.0,1.0,1.0,NaN,...,False,False,True,False,False,False,False,False,True,False
6,1452168870966889341,Vernon,Other Cities,Private room in home,Private room,3.0,4.0,1.0,2.0,65.05,...,False,False,False,False,False,False,False,False,False,True
7,1452186498493578372,Glendale,Other Cities,Entire home,Entire home/apt,8.0,4.0,4.0,4.0,479.00,...,False,False,False,False,False,False,False,False,False,True
8,1453347565305265194,Monterey Park,Other Cities,Entire villa,Entire home/apt,10.0,3.0,4.0,5.0,719.00,...,False,False,False,False,False,False,False,False,False,True
9,1453598695436709692,Marina del Rey,Unincorporated Areas,Entire rental unit,Entire home/apt,2.0,1.0,1.0,1.0,443.00,...,False,False,False,False,False,False,False,False,False,True


## Limitations
Advertised prices are not revenue and availability is not occupancy. Missing price may be systematic. Seasonality, composition, and different scrape windows limit comparisons. instant_bookable is entirely missing. The team must reconcile individual methods before its group submission.